# API Endpoint Testing

Notebook em duas partes:

1. **Requisições simples** — explore cada endpoint manualmente
2. **Testes automatizados** — validações completas + métricas (README)

API REST via **API Gateway** (stage `v1`). Auth: header `x-api-key` (exceto `/health`).

Configure manualmente (opcional):

```bash
export RECOMMENDATIONS_API_BASE_URL="https://<api-id>.execute-api.us-east-1.amazonaws.com/v1"
export RECOMMENDATIONS_API_KEY="<sua-api-key>"
export RECOMMENDATIONS_TEST_USER_ID="u_0231"
export RECOMMENDATIONS_TEST_COLD_START_USER_ID="u_9999"
```

Ou deixe o notebook resolver URL/key via `terraform output` / SSM.

In [ ]:
import json
import os
import subprocess
from dataclasses import dataclass, field
from pathlib import Path

import httpx

TERRAFORM_DIR = (Path("..") / "terraform").resolve()
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
API_STAGE = "v1"
KNOWN_USER_ID = os.getenv("RECOMMENDATIONS_TEST_USER_ID", "u_0231")
COLD_START_USER_ID = os.getenv("RECOMMENDATIONS_TEST_COLD_START_USER_ID", "u_9999")

EXPECTED_METRIC_NAMES = {
    "recommendations_api_requests_total",
    "recommendations_api_errors_total",
    "recommendations_api_cold_start_total",
    "recommendations_api_latency_avg_ms",
}


def _terraform_output(name: str) -> str | None:
    try:
        return subprocess.check_output(
            ["terraform", f"-chdir={TERRAFORM_DIR}", "output", "-raw", name],
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
    except (subprocess.CalledProcessError, FileNotFoundError):
        return None


def normalize_api_base_url(base_url: str) -> str:
    normalized = base_url.rstrip("/")
    if normalized.endswith(f"/{API_STAGE}"):
        return normalized
    if "execute-api" in normalized:
        return f"{normalized}/{API_STAGE}"
    return normalized


def load_api_config() -> tuple[str, str]:
    base_url = os.getenv("RECOMMENDATIONS_API_BASE_URL")
    api_key = os.getenv("RECOMMENDATIONS_API_KEY")

    if not base_url:
        base_url = _terraform_output("recommendations_api_gateway_endpoint")

    if not api_key:
        api_key = _terraform_output("recommendations_api_key")

    if not api_key:
        param_name = _terraform_output("recommendations_api_key_ssm_parameter")
        if param_name:
            import boto3

            ssm = boto3.client("ssm", region_name=AWS_REGION)
            api_key = ssm.get_parameter(Name=param_name, WithDecryption=True)[
                "Parameter"
            ]["Value"]

    if not base_url or not api_key:
        raise RuntimeError(
            "Defina RECOMMENDATIONS_API_BASE_URL e RECOMMENDATIONS_API_KEY "
            "ou aplique o Terraform e configure credenciais AWS."
        )

    return normalize_api_base_url(base_url), api_key


def api_headers(*, with_key: bool = True) -> dict[str, str]:
    headers = {"Accept": "application/json"}
    if with_key:
        headers["x-api-key"] = API_KEY
    return headers


def call_api(
    method: str,
    path: str,
    *,
    json_body: dict | None = None,
    with_key: bool = True,
    timeout: float = 30.0,
) -> httpx.Response:
    url = f"{API_BASE_URL}{path}"
    with httpx.Client(timeout=timeout) as client:
        return client.request(
            method,
            url,
            headers=api_headers(with_key=with_key),
            json=json_body,
        )


def show_response(response: httpx.Response, *, label: str = "") -> httpx.Response:
    """Imprime status e corpo; retorna a response para encadear uso."""
    prefix = f"[{label}] " if label else ""
    print(f"{prefix}{response.request.method} {response.request.url}")
    print(f"{prefix}HTTP {response.status_code}")
    content_type = response.headers.get("content-type", "")
    if "json" in content_type:
        print(json.dumps(response.json(), indent=2, ensure_ascii=False))
    else:
        print(response.text)
    return response


API_BASE_URL, API_KEY = load_api_config()

print(f"API base URL: {API_BASE_URL}")
print(f"Known user:   {KNOWN_USER_ID}")
print(f"Cold start:   {COLD_START_USER_ID}")

## Requisições simples

Execute cada célula abaixo para inspecionar um endpoint. Ajuste `KNOWN_USER_ID` / `COLD_START_USER_ID` na célula de setup se quiser.

### `GET /health` (público, sem API key)

In [ ]:
show_response(call_api("GET", "/health", with_key=False), label="health")

### `GET /recommendation/{user_id}`

In [ ]:
show_response(
    call_api("GET", f"/recommendation/{KNOWN_USER_ID}"),
    label="recommendation",
)

### `GET /recommendations/{user_id}` (alias + cold start)

In [ ]:
print("--- usuário existente ---")
show_response(
    call_api("GET", f"/recommendations/{KNOWN_USER_ID}"),
    label="recommendations",
)

print("\n--- cold start ---")
show_response(
    call_api("GET", f"/recommendations/{COLD_START_USER_ID}"),
    label="cold_start",
)

### `POST /recommendations_filtered`

In [ ]:
show_response(
    call_api(
        "POST",
        "/recommendations_filtered",
        json_body={
            "user_id": KNOWN_USER_ID,
            "limit": 5,
            "exclude_product_ids": ["p_001"],
            "context": {"device": "notebook", "campaign": "manual_test"},
        },
    ),
    label="recommendations_filtered",
)

### `POST /recommendation_filtered` (alias)

In [ ]:
show_response(
    call_api(
        "POST",
        "/recommendation_filtered",
        json_body={"user_id": KNOWN_USER_ID, "limit": 3},
    ),
    label="recommendation_filtered",
)

### `GET /metrics` (Prometheus)

In [ ]:
show_response(call_api("GET", "/metrics"), label="metrics")

## Testes automatizados

Validações com assertions. Execute após explorar os endpoints acima.

In [ ]:
@dataclass
class CheckResult:
    name: str
    passed: bool
    detail: str = ""


@dataclass
class ApiTestReport:
    checks: list[CheckResult] = field(default_factory=list)

    def add(self, name: str, passed: bool, detail: str = "") -> None:
        self.checks.append(CheckResult(name=name, passed=passed, detail=detail))

    def assert_all_passed(self) -> None:
        failures = [check for check in self.checks if not check.passed]
        if failures:
            lines = "\n".join(f"- {item.name}: {item.detail}" for item in failures)
            raise AssertionError(f"Checks failed:\n{lines}")

    def print_results(self) -> None:
        for check in self.checks:
            status = "PASS" if check.passed else "FAIL"
            print(f"[{status}] {check.name}: {check.detail}")


def parse_prometheus_metrics(text: str) -> dict[str, float]:
    metrics: dict[str, float] = {}
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        if "{" in line:
            value = float(line.rsplit(" ", 1)[1])
            if 'quantile="0.5"' in line:
                metrics["recommendations_api_latency_ms_p50"] = value
            elif 'quantile="0.95"' in line:
                metrics["recommendations_api_latency_ms_p95"] = value
            continue
        name, value = line.rsplit(" ", 1)
        metrics[name] = float(value)
    return metrics


def validate_readme_metrics(metrics_text: str, *, min_requests: int) -> ApiTestReport:
    report = ApiTestReport()
    parsed = parse_prometheus_metrics(metrics_text)

    missing = EXPECTED_METRIC_NAMES - set(parsed)
    report.add(
        "prometheus_required_metrics",
        not missing,
        f"missing={sorted(missing)}" if missing else "all required counters/gauges present",
    )

    requests_total = parsed.get("recommendations_api_requests_total", 0.0)
    errors_total = parsed.get("recommendations_api_errors_total", 0.0)
    cold_start_total = parsed.get("recommendations_api_cold_start_total", 0.0)
    p50 = parsed.get("recommendations_api_latency_ms_p50", -1.0)
    p95 = parsed.get("recommendations_api_latency_ms_p95", -1.0)

    report.add(
        "request_count_available",
        requests_total >= min_requests,
        f"requests_total={requests_total}, expected>={min_requests}",
    )
    report.add(
        "error_rate_available",
        "recommendations_api_errors_total" in parsed,
        f"errors_total={errors_total}",
    )
    report.add(
        "cold_start_counter_available",
        cold_start_total >= 1,
        f"cold_start_total={cold_start_total}",
    )
    report.add("latency_p50_available", p50 >= 0, f"p50_ms={p50}")
    report.add("latency_p95_available", p95 >= 0, f"p95_ms={p95}")
    report.add("latency_p95_gte_p50", p95 >= p50, f"p50_ms={p50}, p95_ms={p95}")

    error_rate = (errors_total / requests_total) if requests_total else 0.0
    report.add(
        "error_rate_reasonable",
        error_rate <= 0.5,
        f"error_rate={error_rate:.2%}",
    )
    return report

### Smoke tests dos endpoints

In [ ]:
report = ApiTestReport()

report.add(
    "rest_api_stage_in_base_url",
    API_BASE_URL.endswith(f"/{API_STAGE}"),
    API_BASE_URL,
)

health = call_api("GET", "/health", with_key=False)
report.add("health_status_200", health.status_code == 200, f"status={health.status_code}")
report.add(
    "health_payload",
    health.json().get("status") == "ok",
    str(health.json()),
)

recommendation = call_api("GET", f"/recommendations/{KNOWN_USER_ID}")
rec_body = recommendation.json()
report.add(
    "recommendations_status_200",
    recommendation.status_code == 200,
    f"status={recommendation.status_code}",
)
report.add(
    "recommendations_has_ranked_items",
    rec_body.get("count", 0) > 0 and bool(rec_body.get("recommendations")),
    f"count={rec_body.get('count')}",
)
report.add(
    "recommendations_has_score",
    all("score" in item for item in rec_body.get("recommendations", [])),
    "missing score in at least one item",
)
report.add(
    "recommendations_not_cold_start",
    rec_body.get("cold_start_flag") is False,
    f"cold_start_flag={rec_body.get('cold_start_flag')}",
)

singular = call_api("GET", f"/recommendation/{KNOWN_USER_ID}")
report.add(
    "recommendation_alias_status_200",
    singular.status_code == 200,
    f"status={singular.status_code}",
)
report.add(
    "recommendation_alias_same_user",
    singular.json().get("user_id") == KNOWN_USER_ID,
    str(singular.json().get("user_id")),
)

cold_start = call_api("GET", f"/recommendations/{COLD_START_USER_ID}")
cold_body = cold_start.json()
report.add(
    "cold_start_status_200",
    cold_start.status_code == 200,
    f"status={cold_start.status_code}",
)
report.add(
    "cold_start_flag_true",
    cold_body.get("cold_start_flag") is True,
    f"cold_start_flag={cold_body.get('cold_start_flag')}",
)

filtered = call_api(
    "POST",
    "/recommendations_filtered",
    json_body={
        "user_id": KNOWN_USER_ID,
        "limit": 5,
        "exclude_product_ids": [rec_body["recommendations"][0]["product_id"]],
        "context": {"device": "notebook", "campaign": "endpoint_test"},
    },
)
filtered_body = filtered.json()
report.add(
    "filtered_status_200",
    filtered.status_code == 200,
    f"status={filtered.status_code}",
)
report.add(
    "filtered_respects_limit",
    filtered_body.get("count", 0) <= 5,
    f"count={filtered_body.get('count')}",
)
report.add(
    "filtered_has_detailed_fields",
    all(
        {"product_id", "recommendation_score", "category"} <= set(item)
        for item in filtered_body.get("recommendations", [])
    ),
    "detailed recommendation fields missing",
)
report.add(
    "filtered_context_echo",
    filtered_body.get("context", {}).get("campaign") == "endpoint_test",
    str(filtered_body.get("context")),
)

filtered_alias = call_api(
    "POST",
    "/recommendation_filtered",
    json_body={"user_id": KNOWN_USER_ID, "limit": 3},
)
report.add(
    "recommendation_filtered_alias_status_200",
    filtered_alias.status_code == 200,
    f"status={filtered_alias.status_code}",
)

invalid = call_api("GET", "/recommendations/invalid_user")
report.add(
    "invalid_user_returns_400",
    invalid.status_code == 400,
    f"status={invalid.status_code}",
)

unauthorized = call_api("GET", f"/recommendations/{KNOWN_USER_ID}", with_key=False)
report.add(
    "protected_route_requires_api_key",
    unauthorized.status_code in {401, 403},
    f"status={unauthorized.status_code}",
)

report.print_results()
report.assert_all_passed()
print("Endpoint smoke tests passed.")

### Validação de métricas (`GET /metrics`)

In [ ]:
metrics_response = call_api("GET", "/metrics")
metrics_text = metrics_response.text

print(f"Status: {metrics_response.status_code}")
print(f"Content-Type: {metrics_response.headers.get('content-type')}")
print("\n--- /metrics ---")
print(metrics_text)

metrics_report = validate_readme_metrics(metrics_text, min_requests=6)
metrics_report.add(
    "metrics_status_200",
    metrics_response.status_code == 200,
    f"status={metrics_response.status_code}",
)
metrics_report.add(
    "metrics_prometheus_content_type",
    "text/plain" in (metrics_response.headers.get("content-type") or ""),
    metrics_response.headers.get("content-type", ""),
)

metrics_report.print_results()
metrics_report.assert_all_passed()
print("Metrics validation passed.")